# Late Interaction using ColBert
ColBERT is a late interaction technique to improve search quality.
<br>
In **Late Interaction** using ColBERT :
* embeddings are created for every token in the search query and the document.
* for each query token we find a document token with maximum cosine similarity.
* these maximum similarity (`MaxSim`) scores are aggregated to produce the final document relevance score.

> **NOTE**<br> An honest ColBERT implementation requires one vector per token of each document and MaxSim scoring serverside.<br>
However, turbopuffer doesn't support multi-vector documents today.

Since turbopuffer does not natively support multi-vector documents, a practical ColBERT implementation today is a two-stage approximation:
* dense ANN retrieval followed by 
* client-side MaxSim re-ranking on the top-k candidates.

Key implementation desciions:
* At index time, ColBERT is run on Every document and token vectors vectors are stored as JSON attribute on each document row.
* At query time:    
    * turbopuffer ANN finds top-100 dense vectors fast
    * turbopuffer trturns token_vectors attribute for those 100 docs in the same response (no additional round trip)
    * client computes MaxSim on the pre-computed token matrices.
Advantages of this solution:
* Pre-computed tokens once at index-time, makes every query fast. 
* Single roundtrip to turbopuffer.
* Token vectors trabel with the document, no lookups or joins.
* Storing JSOB blobs is cheaper in turbopuffer's storage model.


### Step 1: Setup
Imports, load .env, initialize turbopuffer client

In [1]:
import os
import json
import turbopuffer
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Access variables
TURBOPUFFER_API_KEY = os.getenv("TURBOPUFFER_API_KEY")

#initialize turbopuffer client
tpuf = turbopuffer.Turbopuffer(
    # API tokens are created in the dashboard: https://turbopuffer.com/dashboard
    api_key=os.getenv("TURBOPUFFER_API_KEY"),
    # Pick the right region: https://turbopuffer.com/docs/regions
    region="gcp-us-central1",
)

#initialize a namespace
ns = tpuf.namespace(f'colbert-late-interaction')

### Step 2: Dataset
Load [Quora Question Paris](https://huggingface.co/datasets/sentence-transformers/quora-duplicates) from HuggingFace

In [2]:
import pandas as pd

df_qpair = pd.read_parquet("hf://datasets/sentence-transformers/quora-duplicates/pair/train-00000-of-00001.parquet")
df_qpair.count()

anchor      149263
positive    149263
dtype: int64

In [3]:
df_qpair.head(100)

,anchor,positive
0,Astrology: I am a Capricorn Sun Cap moon and c...,"I'm a triple Capricorn (Sun, Moon and ascendan..."
1,How can I be a good geologist?,What should I do to be a great geologist?
2,How do I read and find my YouTube comments?,How can I see all my Youtube comments?
3,What can make Physics easy to learn?,How can you make physics easy to learn?
4,What was your first sexual experience like?,What was your first sexual experience?
...,...,...
95,Will Modi win in 2019?,Can Narendra Modi become Prime Minister of Ind...
96,"What exactly is the ""Common Core Initiative/St...",What are the pros and cons of the Common Core ...
97,How do I choose a journal to publish my paper?,Where do I publish my paper?
98,What are your New Year's resolutions for 2017?,What is your creative New Year's resolution fo...


The dataset has ~150K questions pairs - `anchor` and `positive`. <br>We will:
* take a subset of 1000 questions pairs
* use `positive` to build corpus, create dense embeddings + `token_vector` attribute in turbopuffer
* use a few `anchor` questions to compare the search performance of **dense retreival** vs **late interaction**

Succes metric would be "did the matching positive for given anchor rank #1".

In [4]:
df_corpus = df_qpair['positive'][:1000]
print(f"Dataframe shape: {df_corpus.shape}")
print(f"Corpus length is {df_corpus.count()} \nCorpus Sample:")
df_corpus.head()

Dataframe shape: (1000,)
Corpus length is 1000 
Corpus Sample:


0    I'm a triple Capricorn (Sun, Moon and ascendan...
1            What should I do to be a great geologist?
2               How can I see all my Youtube comments?
3              How can you make physics easy to learn?
4               What was your first sexual experience?
Name: positive, dtype: str

### Step 3: Embedding
Load ColBERT model via fastembed, generate dense vector and token vectors per document.<br>
> We choose [`fastembed`](https://huggingface.co/ferrisS/harrier-oss-v1-270m-fastembed) because it is a lightweight embedding library with native late interaction support, and runs on CPU.

In [5]:
from fastembed import LateInteractionTextEmbedding

model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")

In [6]:
sample = df_corpus[0]
vector = list(model.embed([sample]))[0]

print(sample)
print(vector.shape)

I'm a triple Capricorn (Sun, Moon and ascendant in Capricorn) What does this say about me?
(26, 128)


In [7]:
# one single vector to represent the whole document
dense_vector = vector.mean(axis=0)

# matrix of 128-dimensional vector of all tokens in the document.
token_vectors = vector.tolist()

print(f"Dense vector shape: {dense_vector.shape}")
print(f"Token matrix: {len(token_vectors)} tokens x {len(token_vectors[0])} dims")

Dense vector shape: (128,)
Token matrix: 26 tokens x 128 dims


In [8]:
# Loop over all 1000 corpus questions, generate dense and token vectors.

records = []

for i in range(len(df_corpus)):
    text = df_corpus[i]
    embedding = list(model.embed([text]))[0]

    # one single vector to represent the whole document
    dense_vector = embedding.mean(axis=0).tolist()

    # matrix of 128-dimensional vector of all tokens in the document.
    token_vectors = embedding.tolist()

    record = {
        "id": i + 1,
        "vector": dense_vector,
        "token_vectors": json.dumps(token_vectors), # serialize to string
        "text": text
    }

    records.append(record)
print(f"Embedded {len(records)} documents")



Embedded 1000 documents


### Step 4: Index
Upsert to turbopuffer:
* `dense_vector` column for ANN, 
* `token_vectors` as JSON attribute, 
* `text` as attribute
> `token_vectors` is a non-filterable attribute,, so paying for a filterable index on them would be wasteful. <br>
 Set `filterable: false` in the schema for a 50% storage cost discount.

In [9]:
# Upsert documents with vectors and attributes
ns.write(
    upsert_rows=records,
    distance_metric='cosine_distance',
    schema={
        "token_vectors": {"type": "string", "filterable": False},
        "text": {"type": "string", "filterable": False}
    }
)

print(f"Upserted {len(records)} documents to turbopuffer")

Upserted 1000 documents to turbopuffer


### Step 5: Query: Dense only
Run a test query, time it, show top 5 results

In [14]:
import time

def dense_search(query, top_k=5):
    query_embedding = list(model.embed([query]))[0]
    query_vector = query_embedding.mean(axis=0).tolist()

    start_ts = time.perf_counter()
    results = ns.query(
        rank_by=["vector", "ANN", query_vector],
        limit=top_k,
        include_attributes=['text', 'token_vectors']
    )
    end_ts = time.perf_counter()
    latency = (end_ts - start_ts) * 1000

    return results.rows, latency

# dense search test
results, latency = dense_search(df_qpair["anchor"][2]) #use an anchor question from the original question 

print(f"Original question pair: {df_qpair.iloc[2]}")
print(f"Latency: {latency:.1f}ms")
for row in results:
    print(f"Distance: {row['$dist']:.4f} | {row.text}")

Original question pair: anchor      How do I read and find my YouTube comments?
positive         How can I see all my Youtube comments?
Name: 2, dtype: str
Latency: 293.1ms
Distance: 0.0955 | How can I see all my Youtube comments?
Distance: 0.2524 | How can I see who viewed my video I just posted on Instagram?
Distance: 0.4082 | What should I do if someone has posted porn video under my name?
Distance: 0.4367 | How do I make money through YouTube?
Distance: 0.4367 | How do I make money through YouTube?


### Step 6: Query: Late Interaction
Same query, ANN top-100, client-side MaxSim re-rank, time it, show top 5 results

In [11]:
import numpy as np

#MaxSim function
def maxsim(query_tokens, doc_tokens):
    query_tokens = np.array(query_tokens) #(Q, 128) matrix
    doc_tokens = np.array(doc_tokens) #(D, 128) matrix
    scores = np.dot(query_tokens, doc_tokens.T) #score is (Q, D) matrix
    max_similarities = scores.max(axis=1) #max similarity score for each query token against the whole document.
    maxim_score = max_similarities.sum() #final aggregated maxim socre of the query against the document

    return maxim_score 

#Full ColBERT search function
def colbert_search(query, top_k=5):
    query_embedding = list(model.embed([query]))[0]

    #stage 1 - dense retreival of top 100 ANN documents
    candidates, l = dense_search(query, top_k=100)

    #stage 2 - rerank with MaxSim
    start_ts = time.perf_counter()
    scores = []
    for candidate in candidates:
        doc_tokens = json.loads(candidate.token_vectors)
        score = maxsim(query_embedding, doc_tokens)
        scores.append((score, candidate.text))

    scores.sort(key=lambda x: x[0], reverse=True)
    end_ts = time.perf_counter()
    latency = (end_ts - start_ts) * 1000

    return scores[:top_k], latency

# colbert search test
results, latency = colbert_search(df_qpair["anchor"][10])

print(f"Original question pair: {df_qpair.iloc[10]}")
print(f"Colbert search latency: {latency:.1f}ms")
for score, text in results:
     print(f"Score: {score:.4f} | {text}")


Original question pair: anchor      What are some special cares for someone with a...
positive    How can I keep my nose from getting stuffy at ...
Name: 10, dtype: str
Colbert search latency: 61.2ms
Score: 13.4563 | How can I keep my nose from getting stuffy at night?
Score: 10.9009 | Why some people sweat excessively in their armpit but it has no odorous stinky smell?
Score: 10.7436 | How do you apply Vicks Vapor rub to treat a stuffy nose?
Score: 9.6989 | Who is your favourite character in the TV series The Walking Dead? Why?
Score: 9.4430 | What, if anything, is too serious to be joked about in life?
